# Lab 1 Agent-Dev (build the pipeline yourself, with help)

This is the **developer stage** of a two-stage exercise: an **agent**<sub>4</sub> helps you *write* the pipeline's tools, one file at a time.

New to this developing and this vocabulary? See `agent_dev/GLOSSARY.md`. A small number like the one above points back to it.

## Before you start, a few terms

"Agent" gets used loosely, so let's be precise. (Full list:
`agent_dev/GLOSSARY.md`; a number like this<sub>1</sub> points back to
it.)

- **The LLM**<sub>1</sub> is just the model (Gemini or Groq): a
  **transformer**<sub>3</sub> that predicts the next **token**<sub>2</sub>
  from whatever text it's given. No memory between calls, no way to
  read a file or run code by itself.
- **A tool**<sub>5</sub> is what turns a plain chatbot into something
  that acts: a Python **function**<sub>37</sub> with a name, typed
  parameters, and a docstring, wrapped so the model can request it by
  name. The model never runs a tool itself. It only predicts "call
  this tool with these arguments," and the harness actually executes
  it.
- **The harness**<sub>6</sub> is what turns that into an agent. It
  resends the whole conversation every call, notices when the model
  asks to run a tool, runs it, and feeds the result back in. Here
  that's `graph_core.py` (a LangGraph **state machine**<sub>28</sub>),
  plus the chat widget, already built for you.
- **LangGraph**<sub>7</sub> is the library the harness is built with.
  Instead of hand-writing the send-call-run-repeat loop, you describe
  it as a graph: named steps (**"nodes"**<sub>8</sub>) connected by
  **edges**<sub>9</sub> that say what runs next. `graph_core.py` has
  three: `agent` (calls the model), `tool_node` (runs the tool),
  `discuss` (comments on the result). That's why "one tool call, then
  a comment, then stop" always holds, no matter what you or the model
  try.
- **Memory**<sub>10</sub> here just means the conversation list the
  harness resends every call, gone when the kernel restarts. A
  longer-lived system might use a database or a
  **vector store**<sub>11</sub> instead.

The only thing you're responsible for is what goes on inside a tool:
the function itself.

**`@tool`**, from `langchain_core.tools`, is the
**decorator**<sub>19</sub> every tool starts with. It builds a
`.args_schema` from the function's **type hints**<sub>21</sub> and
**docstring**<sub>22</sub>, and the model calls it correctly using only
that **schema**<sub>20</sub>; it never reads the function body. Skip
a type hint and the model has to guess that parameter's type; skip
docstring detail and it won't know when to call the tool.
`premade_tools/data_loading_tools.py`'s `load_dataset_tool` shows the
pattern.

**Why types matter here specifically:** a vague type makes the model
guess silently. Your code runs, just on the wrong input, no error.
An incompatible type can make the whole call fail outright: Gemini
rejects a fixed-length tuple like `Tuple[int, int]` this way.
`agent_dev/BACKEND_SCHEMA_NOTES.md` lists exactly which types are safe
on both backends. The agent is instructed to read that page itself
before typing any parameter, using its one tool that reaches outside
your workspace (`read_spec_file_tool`, allowed for exactly this page
and two others, nothing more). Worth reading yourself too, so you'd
recognize the problem if you ever saw it in a raw error message.

Four kinds of tools show up in this course: **read-only** (just
reports something, e.g. `inspect_data_tool`), **pipeline** (changes
session state, e.g. `build_dtm_tool`), **meta** (manages what the
agent can do, e.g. `propose_file_tool`), and **real-world** (acts
outside the machine, not used here, but the mechanism is the same).

You're building the eighteen tools that aren't `load_dataset_tool` or
`visualize_result_tool`.

## Your three reference files

Everything you need to build these tools correctly lives in
`agent_dev/`, in three short files. Read all three before you start:

- **`TOOLS_PIPELINE_BREAKDOWN.md`**: the exact shape every tool you
  build has to match, what it should return, what the shared `session`
  object holds, plus extra detail for the trickier tools. Read this
  before asking the agent to build anything.
- **`TEST_FIXTURE.md`**: a tiny, 8-document practice dataset with the
  correct answer for every tool, worked out by hand, not by running
  your own code. Use it to test what you build against real numbers,
  not guesses.
- **`BACKEND_SCHEMA_NOTES.md`**: one specific way of writing a
  parameter that breaks on Gemini (not Groq). Read this before you or
  the agent decide how to type a tool's parameters.

Why read them yourself, not just leave it to the agent: the agent can
only open these three when you point it at the right one by name, and
never on its own just because it seems relevant. Reading them yourself
means you can actually check what the agent tells you against the real
thing, instead of taking its word for it.

## How writing works

Two steps, two messages:
1. You ask for code. The agent proposes it and shows a
   **diff**<sub>26</sub> or preview. **Nothing is written yet.**
2. Only after you approve in your *next* message does it write the
   file. This is a **structural**<sub>29</sub> rule, not something you
   or the model can skip: propose and write can never happen in the
   same turn.

**The agent can't see a finished reference implementation**<sub>33</sub>.
Its file tools are confined to your own **workspace**<sub>25</sub>
(`agent_dev_workspace/`), which starts empty. See
`agent_dev/TOOLS_PIPELINE_BREAKDOWN.md` for the shape your code needs
to match.

Same loop as a diagram, one tool at a time:

```mermaid
flowchart TD
    A["You ask for one specific tool,\nor push back on a plan first"] --> B["Agent proposes the code\n(nothing written yet)"]
    B --> C{"You review it"}
    C -- "request changes" --> B
    C -- "approve" --> D["Agent writes the file"]
    D --> E["You ask for a test against\nthe fixture's known answers"]
    E --> F["Agent proposes the test\n(same review gate)"]
    F --> G{"You review, approve"}
    G -- "approve" --> H["Agent writes the test"]
    H --> I["You ask it to run the test"]
    I --> J["Agent actually runs it\n(real pass/fail, not a guess)"]
    J -- "fails" --> B
    J -- "passes" --> K["Move to the next tool"]
    K --> A
```

Every arrow back to "You" is a turn the graph forces to stop and wait
for your next message. No shortcuts: no "written" without your
approval, no "passes" without `run_tests_tool` actually running.

## Which tools you're actually building

`agent_dev/TOOLS_PIPELINE_BREAKDOWN.md` has the full table of all
eighteen tools and the exact **interface**<sub>23</sub> your code
needs to match: the `session` object, the `make_tools(session)`
**contract**<sub>24</sub>, all of it. Read it once before you start.

Two exceptions: `load_dataset_tool`
(`premade_tools/data_loading_tools.py`) and `visualize_result_tool`
(`premade_tools/visualization_tools.py`, which just renders a bar
chart of a stored result). `load_dataset_tool` is worth reading first,
since it's the pattern your own eighteen tools need to follow:

```python
@tool
def load_dataset_tool(
    categories: list[str] = None,
    file_path: str = None,
    text_column: str = None,
    label_column: str = None,
) -> dict:
    """Load a dataset. Two mutually exclusive modes, pass exactly one:
    `categories` for built-in 20-newsgroups, or `file_path` (plus
    `text_column`, `label_column`) for a local CSV.

    Args:
        categories: newsgroup names, e.g. ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]. Mode 1 only.
        file_path: path to a local CSV, relative to the repo root. Mode 2 only.
        text_column: CSV column with the document text. Required with file_path.
        label_column: CSV column with the category/label. Required with file_path.
    """
    ...
```

The type hints, the two-modes docstring, the
`session.store_result(...)` call at the end: that's the shape to
copy. The only thing off-limits is a finished answer for the tools
*you're* building.

**How this one would have been built, if it weren't done for you
already:** the same propose-then-approve conversation you're about
to have.

> **You:** Let's build load_dataset_tool in tools_data.py. It needs to
> support two ways of loading data: the built-in 20-newsgroups subset
> by category, or a local CSV with a text column and a label column.
> Both should end up in the same shape afterward so nothing downstream
> has to care which one was used.
>
> **Agent:** Here's the plan: one function, two mutually exclusive
> modes selected by which arguments are passed: `categories` for
> 20-newsgroups, `file_path` plus `text_column`/`label_column` for a
> CSV. Both branches build a DataFrame with the same `text`,
> `category_name`, and `category` columns, so `session.dataframe` looks
> identical either way. Want me to propose it?
>
> **You:** Yes, propose it.
>
> **Agent:** *(proposes the code, explains the two branches, the
> normalization step, and the `session.store_result(...)` call)*
>
> **You:** Looks right. Write it.

Same propose, discuss, approve, write sequence as "How writing works"
above, just already run once, as your reference.

## What kind of help you get, tool by tool

Not every tool comes with the same amount of scaffolding. Twenty
total, three different levels:

- **2 provided, already built**: `load_dataset_tool` and
  `visualize_result_tool`. Nothing to build here; read them for
  reference.
- **16 with a full contract and a known answer**: everything else
  except the two listed below. `TOOLS_PIPELINE_BREAKDOWN.md` defines
  exactly what each should return, and `TEST_FIXTURE.md` has a
  worked-out correct answer to test against.
- **2 with neither, on purpose**: `inspect_data_tool` and
  `binarize_labels_tool`. No defined contract, no known answer in the
  fixture. You still build these with the agent's help, the same
  propose-approve-write process as any other tool, but you're the one
  who decides what they should return and what a reasonable test
  looks like, using what you've picked up from the other sixteen.
  They were picked for this specifically because nothing else in the
  pipeline depends on their exact shape, and because they're simple
  enough that you shouldn't get stuck: if you can build the other
  sixteen, you can reason through these two on your own.

If you ask the agent to test one of these last two the way you tested
the others (against a known-answer number), it should tell you plainly
that no known answer exists for that tool, not invent one. That's the
correct behavior, not a bug.

## Work from the spec, not from memory

Three spec files, all in `agent_dev/`: `TOOLS_PIPELINE_BREAKDOWN.md`
(the interface contract, the `session` object, the `make_tools(session)`
shape), `TEST_FIXTURE.md` (the fixture data and known answers), and
`BACKEND_SCHEMA_NOTES.md` (which parameter types are safe on both
backends). Read all three yourself before you start.

The agent has exactly one tool that reaches outside your own workspace:
`read_spec_file_tool`, and it can only read these same three files by
exact name, nothing else, no reference implementation anywhere in
reach. It's instructed to call it for the relevant spec before
proposing a tool's code, a test's expected numbers, or a parameter's
type, if it hasn't already this conversation. That's what makes this
**"spec-driven development"**<sub>34</sub> actually true, not just a
name: agree on a written spec, then have the agent go read the current
text of it and build against that, rather than recalling it from
whatever it was told when it was set up.

**You should still be the one who names the spec file, every time it
applies, not just trust that it will check on its own.** The agent is
instructed to call `read_spec_file_tool` before proposing code, a
test, or a parameter type, whether or not you name the file. But
instruction-following isn't guaranteed on every single turn, the same
way any rule in this agent's instructions could in principle be missed
once, and describing what you want in your own words gives it more
room to skip the fetch and answer from its general sense of the
assignment instead. Say "Build variance_filter_tool matching
TOOLS_PIPELINE_BREAKDOWN.md's contract," not "build me the variance
filter tool." Say "Test it against known answer 5 in TEST_FIXTURE.md,"
not the number itself. Naming the file is both a nudge that makes the
fetch more likely and something you can point back to if it doesn't
happen: a proposal that doesn't match what the named file actually
says is a clear sign it skipped the read. Skipping the read is exactly
the situation this whole setup exists to avoid, since a guessed
contract or number is a **hallucination**<sub>16</sub> waiting to
happen, not a shortcut that happens to still work; a plausible-sounding
contract or number that's actually wrong is worse than an honest "I'm
not sure," since it looks just as confident either way.

**Then actually make the agent go read it, don't just assume it
will.** If you're ever unsure whether it did, or a proposal looks like
it's guessing at a contract or a number rather than citing one, ask it
directly: "did you read TOOLS_PIPELINE_BREAKDOWN.md for this?" A
correctly-behaving agent should be able to say yes and point to what it
found there; if it can't, don't accept the proposal, ask it to actually
read the spec first.

## Why testing here means testing against a fixture

**A test**<sub>35</sub>: code that runs your function on a known
input and checks the output with **`assert`**<sub>39</sub>. It's a
real file, same as any tool file: propose it, approve it, and it's
written to disk.

**Why test at all:** a tool can run cleanly, return something
**JSON-serializable**<sub>32</sub>, and still give the wrong answer.
"Doesn't crash" and "is correct" aren't the same claim. A test catches
the second one, now, before it affects anything else.

**A fixture**<sub>36</sub>: a small, fixed input with a correct answer
worked out some other way, not by running your own code. The real
dataset is too big for this, and its "correct" answer usually isn't
known ahead of time. A fixture is small enough to check by hand, so a
test built on it has something real to compare against.

Two things worth knowing when the code *and* the test both come from
an agent:

Ask it to write both with no fixture in mind, and it invents its own
input and its own expected answer, straight from the code it just
wrote. That test can never fail: whatever the code computes becomes
"correct" by definition, even on a broken tool.

And an LLM can simply say a test passed without running it. It's
**stochastic**<sub>17</sub>, not **deterministic**<sub>18</sub> like a
real test, and it can state something false confidently, a
**hallucination**<sub>16</sub>, not a lie, just wrong. That's why
`run_tests_tool` actually launches **pytest**<sub>38</sub> as a real
**subprocess**<sub>30</sub> and checks its real
**exit code**<sub>31</sub>, instead of asking the agent whether it
thinks the test passed.

Read `agent_dev/TEST_FIXTURE.md` before your first test. It has the
real fixture data and exact numbers to test against, worked out
independently, never by asking the same agent that wrote a tool what
its own tool should return. Known answers exist for sixteen of the
eighteen tools you build (`visualize_result_tool` is provided, out of
scope); `inspect_data_tool` and `binarize_labels_tool` are the two
deliberate exceptions, covered in "What kind of help you get, tool by
tool" above.

## 0. Setup: your name, a backend, and its API key

Enter your name and student ID. Both get written into your session
log's header so grading can match your log to you.

Two **backends**<sub>12</sub>: `groq` or `gemini`. **Groq is the
recommended default**, faster on tool-calling turns (Gemini can take
up to a minute, sometimes fails outright). Use Gemini as a fallback
once Groq's **rate limit**<sub>13</sub> runs out. Set up `config/.env`
with your key first (see `config/.env.example`).

Got more than one key? **Turn on multi-provider
fallback**<sub>14</sub>: if your backend hits a rate limit
mid-session, it automatically retries with the next one instead of
leaving you stuck. Worth it: both providers' free tiers only reset
once a day (Groq at midnight UTC, Gemini at midnight Pacific).

Type `/status` in the chat at any point to see which of your keys
are still active and which have hit a rate limit this session,
handled entirely by the chat widget itself, not the agent, so it
never uses a turn or shows up in your graded log.

Run the cell below, fill it in, and click **Confirm**.

In [3]:
from agent_pipeline.backend_widget import select_backend

backend_choice = select_backend()

## 1. What to try

The chat launches below. Work through this in two phases.

Each numbered line below is tagged **[PROMPT]** or **[INSTRUCTION, not
a prompt]**. A `[PROMPT]` line is close to what you'd actually type.
An `[INSTRUCTION, not a prompt]` line tells you what to do or ask in
your own words; there's no fixed line to type there.

**Phase 1: try the test-and-run mechanism on something that already
works.** `load_dataset_tool` is already built and correct, so testing
it first shows you `run_tests_tool` without wondering whether a
failure is your fault or the test's.

1. **[PROMPT]** `What can you help me build?` (just conversation, no tool call)
2. **[PROMPT]** `Can you build a test for load_dataset_tool? It should load agent_dev/sample_fixture.csv and check that it produces exactly 8 documents, split 4 and 4 between catA and catB.`
3. **[PROMPT]** `That looks good, write it`
4. **[PROMPT]** `Does it pass? Run it.` (should pass immediately: you're confirming the mechanism, not debugging your own code)

Roughly:

> **You:** Can you build a test for load_dataset_tool? It should load
> agent_dev/sample_fixture.csv and check that it produces
> exactly 8 documents, split 4 and 4 between catA and catB.
>
> **Agent:** Here's a proposed `tests/test_load_dataset_tool.py`: it
> loads the fixture through `load_dataset_tool`, then asserts
> `n_documents == 8` and `counts_per_category == {"catA": 4, "catB": 4}`.
> Want me to write it?
>
> **You:** Yes, write it.
>
> **Agent:** Written. Ask me to run it whenever you're ready.
>
> **You:** Does it pass? Run it.
>
> **Agent:** Ran it with pytest: 1 passed. `load_dataset_tool` loaded
> exactly 8 documents, 4 catA and 4 catB, matching what the test
> expected.

Worth trying on purpose: asking for two things in one message doesn't
get you two things done.

> **You:** Write the test and run it, both right now.
>
> **Agent:** *(proposes the test, same as any first request, since
> nothing's approved yet)* Approve it and I'll write it, then ask me
> to run it separately. That's still two more turns.

Not caution: writing itself is already two turns (propose, approve)
before running can start.

> **You:** Does it pass? *(asked before anything's proposed)*
>
> **Agent:** There's no test written for `load_dataset_tool` yet. Want
> me to propose one first?

`run_tests_tool` only runs a file that actually exists; asking too
early just gets an honest "nothing to run yet."

**Phase 2: build one of your own eighteen, the same way, this time
actually using the spec.**

**One tool per message.** The agent runs at most one tool each time
you send a message. That's on purpose: it keeps you directing each
step, and it's what stops the agent from proposing and writing a file
in the same turn without your approval. So ask for one thing at a
time. If you ask for two anyway, it does the first and then tells you
the next step it wants ("Next it wants to run ..."); reply "go ahead"
to run it.

5. **[PROMPT]** `Before we build anything, read TOOLS_PIPELINE_BREAKDOWN.md and keep its tool table as the spec for everything we do from here.` Then, as its own separate message: **[PROMPT]** `Now read TEST_FIXTURE.md and keep its known answers as the spec for every test we write.` (two messages, not one, since reading each spec file is its own tool call; this anchors the agent on the real spec, not its general sense of the assignment)
6. **[PROMPT]** `Let's build build_dtm_tool in tools_dtm.py, matching TOOLS_PIPELINE_BREAKDOWN.md's contract. Before you propose any code, explain your plan and why.` (build_dtm_tool is a good first pick, since it only needs load_dataset_tool's output)
7. **[PROMPT]** `Why that approach and not some other way?`, if anything in the plan isn't obvious
8. **[PROMPT]** `Did you read TOOLS_PIPELINE_BREAKDOWN.md for that, or BACKEND_SCHEMA_NOTES.md for ngram_range's type?` (checks that it actually called read_spec_file_tool rather than recalling the contract from general sense; a correctly-behaving agent should be able to point to what it found)
9. **[PROMPT]** `That plan sounds good. Propose it, decorated with @tool and a type hint on every parameter.`
10. **[INSTRUCTION, not a prompt]** Review the diff, ask questions or request changes
11. **[PROMPT]** `That looks good, please write it`
12. **[PROMPT]** `Now build a test for build_dtm_tool using agent_dev/sample_fixture.csv, checking it against known answer 4 in TEST_FIXTURE.md.` (naming the known answer gets you exact numbers, not a typo waiting to happen)
13. **[PROMPT]** `That looks good, write it`
14. **[PROMPT]** `Does it pass? Run it.` (a real failure is possible now, and that's the test doing its job)
15. **[PROMPT]** `Show me what's in my workspace so far` (list_workspace_files_tool)
16. **[PROMPT]** `Read back tools_dtm.py` (read_workspace_file_tool)
17. **[INSTRUCTION, not a prompt]** Ask for the next tool, e.g. `Now let's build the variance filter in tools_filtering.py, matching TOOLS_PIPELINE_BREAKDOWN.md's contract`, and repeat

Two things the agent will *not* stop you from doing:
- `Just write the whole file, skip the explanation`: the agent will
  actually do this if you ask. Still a bad idea, since your log is
  what gets graded, and grading specifically checks for this.
- `Can you build me a load_dataset_tool?`: it'll propose one even
  though it already exists. Wasted effort, not a mistake it catches.

One thing that *is* still enforced: the agent never writes something
you didn't just approve. That's the propose/approve
**gate**<sub>27</sub> itself, not a rule about shortcuts.

**If the agent's reply is just "(No response came back for that...)"**,
that's not a bug, and not the session getting stuck. Behind the scenes,
an empty reply is automatically retried several more times,
switching to a different key or backend each time if you have more
than one configured, before this message ever shows, so seeing it at
all means every one of those attempts came back with nothing usable.
The tool call itself, if there was one that turn, already ran either
way; only the follow-up comment is missing. Just ask it to summarize,
or move to your next question.

In [5]:
from agent_dev.dev_session_state import DevSessionState
from agent_dev.graph import build_graph
from agent_pipeline.chat_widget import launch_chat

if "backend" not in backend_choice:
    raise RuntimeError("Pick a backend in the cell above and click Confirm before running this one.")
LLM_BACKEND = backend_choice["backend"]
MULTI_PROVIDER_MODE = backend_choice["multi_provider"]
STUDENT_NAME = backend_choice["student_name"]
STUDENT_ID = backend_choice["student_id"]

session = DevSessionState(workspace_dir="agent_dev_workspace")

if MULTI_PROVIDER_MODE:
    from agent_pipeline.llm_fallback import build_fallback_llm
    llm, active_chain = build_fallback_llm(LLM_BACKEND, temperature=0)
    print(f"Fallback chain (in order): {active_chain}")
else:
    from agent_pipeline.llm_backends import build_llm_for_backend
    llm = build_llm_for_backend(LLM_BACKEND, temperature=0)

app = build_graph(session, llm)

graph_state, logger = launch_chat(
    app, session, log_path="session_logs/agent_dev_session.jsonl.enc",
    student_name=STUDENT_NAME, student_id=STUDENT_ID, track="agent-dev",
    placeholder="Ask the agent to help you build a tool... (or type /status to see key rotation status)",
    llm=llm,
)

## 2. Submitting your work

Your session is logged turn by turn to
`session_logs/agent_dev_session.jsonl.enc`, encrypted as it's written.
You can't read your own log back. Your built tools live in
`agent_dev_workspace/`. **Both need to be in your pushed fork.**

**Important: both of those folders are gitignored by default**, so
they can hold your own local test runs without cluttering this shared
repo. That means a plain `git add .` will silently skip them, not
push an empty-looking submission on purpose, just because they're
never staged in the first place. Before your final commit, force-add
them explicitly:
```
git add -f agent_dev_workspace session_logs
git commit -m "agent-dev submission"
git push
```
Run `git status` afterward and confirm both show as staged/committed,
not just `git add .` on its own.

In [3]:
import os

log_path = "session_logs/agent_dev_session.jsonl.enc"
if os.path.exists(log_path):
    print(f"Log file present: {log_path} ({os.path.getsize(log_path)} bytes)")
else:
    print("No log yet. Send at least one message in the chat above first.")

Log file present: session_logs/agent_dev_session.jsonl.enc (646357 bytes)
